In [1]:
import pandas as pd

mergeddf = pd.read_csv("tthermo_table.csv")
proteinid = mergeddf["proteinid"].tolist()

In [2]:
import requests
import time
import json
divalentcations = {"Mg(2+)", "Zn(2+)", "Mn(2+)", "Ca(2+)", "Fe(2+)", "Co(2+)", "Ni(2+)", "Cu(2+)"}
results = []

In [3]:
for i, protein in enumerate(proteinid):
    if i % 100 == 0:
        print(f"Processing {i}/{len(proteinid)}")
    url = f"https://rest.uniprot.org/uniprotkb/{protein}.json"
    response = requests.get(url)
    data = response.json()
    
    cofactornames = []
    for comment in data.get("comments", []):
        if comment.get("commentType") == "COFACTOR":
            for cof in comment.get("cofactors", []):
                cofactornames.append(cof["name"])
    
    hascovalent = False
    hasdivalent = False
    for name in cofactornames:
        if name in divalentcations:
            hasdivalent = True
        else:
            hascovalent = True
    
    results.append({
        "accession_id": protein,
        "cofactor_names": cofactornames,
        "has_covalent": hascovalent,
        "has_divalent": hasdivalent
    })
    

cofactor_df = pd.DataFrame(results)

Processing 0/1225
Processing 100/1225
Processing 200/1225
Processing 300/1225
Processing 400/1225


SSLError: HTTPSConnectionPool(host='rest.uniprot.org', port=443): Max retries exceeded with url: /uniprotkb/Q72IN1.json (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:1016)')))

In [4]:
print(len(results))

419


In [16]:
accessionids = [r["accession_id"] for r in results]
print(f"Total entries: {len(accessionids)}")
print(f"Unique entries: {len(set(accessionids))}")

Total entries: 0
Unique entries: 0


In [15]:
seen = set()
nresults = []
for r in results:
    if r["accession_id"] not in seen:
        seen.add(r["accession_id"])
        cleanresults.append(r)

results = nresults
print(f"Deduplicated: {len(results)}")

Deduplicated: 0


In [13]:
print(len(results))

267


In [6]:
done = set(r["accession_id"] for r in results)

for i, protein in enumerate(proteinid):
    if protein in done:
        continue
    if i % 100 == 0:
        print(f"Processing {i}/{len(proteinid)}")
    url = f"https://rest.uniprot.org/uniprotkb/{protein}.json"
    try:
        response = requests.get(url)
        data = response.json()
    except Exception as e:
        print(f"Error on {protein}: {e}")
        time.sleep(5)
        continue
    
    cofactornames = []
    for comment in data.get("comments", []):
        if comment.get("commentType") == "COFACTOR":
            for cof in comment.get("cofactors", []):
                cofactornames.append(cof["name"])
    
    hascovalent = False
    hasdivalent = False
    for name in cofactornames:
        if name in divalentcations:
            hasdivalent = True
        else:
            hascovalent = True
    
    results.append({
        "accession_id": protein,
        "cofactor_names": cofactornames,
        "has_covalent": hascovalent,
        "has_divalent": hasdivalent
    })
    
    time.sleep(1)

cofactor_df = pd.DataFrame(results)

Processing 500/1225
Processing 600/1225
Processing 700/1225
Processing 800/1225
Error on Q72IX5: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
Processing 900/1225
Processing 1000/1225
Processing 1100/1225
Processing 1200/1225


In [7]:
cofactor_df = pd.DataFrame(results)
print(f"Total: {len(cofactor_df)}")
print(f"Covalent: {cofactor_df['has_covalent'].sum()}")
print(f"Divalent: {cofactor_df['has_divalent'].sum()}")
print(f"Both: {(cofactor_df['has_covalent'] & cofactor_df['has_divalent']).sum()}")
print(f"Neither: {(~cofactor_df['has_covalent'] & ~cofactor_df['has_divalent']).sum()}")

Total: 1224
Covalent: 132
Divalent: 141
Both: 18
Neither: 969


In [12]:
response = requests.get("https://rest.uniprot.org/uniprotkb/Q72IX5.json")
data = response.json()
thiscofactor = []
for comment in data.get("comments", []):
    if comment.get("commentType") == "COFACTOR":
        for cof in comment.get("cofactors", []):
            thiscofactor.append(cof["name"])
print(thiscofactor)

[]


In [13]:
results.append({
    "accession_id": "Q72IX5",
    "cofactor_names": [],
    "has_covalent": False,
    "has_divalent": False })
cofactor_df = pd.DataFrame(results)
print(len(cofactor_df))

1225


In [15]:
cofactor_df.to_csv("cofactorflags.csv", index=False)